In [0]:
import torch
print(torch.__version__)

import warnings
warnings.filterwarnings("ignore")

In [0]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [0]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

In [0]:
df.shape

In [0]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [0]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [0]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [0]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [0]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

In [0]:
X_train_tensor.shape

In [0]:
y_train_tensor.shape

In [0]:
import torch.nn as nn
class Model(nn.Module):

    def __init__(self, num_features):
        
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out

In [0]:
learning_rate = 0.1
epochs = 25

In [0]:
loss_function = nn.BCELoss()

In [0]:
model = Model(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)
# define loop
for epoch in range(epochs):
    y_pred = model(X_train_tensor)
    loss = loss_function(y_pred, y_train_tensor.reshape(-1, 1))
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # print loss in each epoch
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')
    

In [0]:
model.bias

In [0]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')